In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_33.pickle

In [4]:
%%RecordEvent
%%time
### cell 33 ###

### cell 33 optimized ###

# define the grouping of framework columns
framework_groups = {
    'TensorFlow/Keras': ['TensorFlow ', 'Keras '],
    'PyTorch/PyTorch Lightning/Fast.ai': ['PyTorch ', 'PyTorch Lightning ', 'Fast.ai '],
    'Xgboost/LightGBM/CatBoost': ['Xgboost ', 'LightGBM ', 'CatBoost ']
}

# subset, rename and drop rows with all NULLs (on GPU via cudf)
ml_frameworks_df_2022_cell45 = (
    responses_df_2022_cell10
    .filter(like=question_of_interest_cell44, axis=1)
    .rename(columns=lambda col: col.split('-')[-1].lstrip())
    .dropna(how='all')
)

# build all new grouped columns in one vectorized .assign, then drop originals
ml_frameworks_df_2022_cell45 = (
    ml_frameworks_df_2022_cell45
    .assign(
        **{
            grp: ml_frameworks_df_2022_cell45[cols]
                    .notna()
                    .any(axis=1)
                    .map({True: grp})
            for grp, cols in framework_groups.items()
        }
    )
    .drop(columns=[c for cols in framework_groups.values() for c in cols])
)[['Scikit—learn '] + list(framework_groups.keys())]

ml_frameworks_df_2022_cell45.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14531 entries, 1 to 23996
Data columns (total 4 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Scikit—learn                       11403 non-null  object
 1   TensorFlow/Keras                   8850 non-null   object
 2   PyTorch/PyTorch Lightning/Fast.ai  5516 non-null   object
 3   Xgboost/LightGBM/CatBoost          4931 non-null   object
dtypes: object(4)
memory usage: 567.6+ KB
CPU times: user 262 ms, sys: 12.4 ms, total: 274 ms
Wall time: 281 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_33_try_2.pickle